# Лекция 12. Защита: adversarial training и сертифицированная устойчивость

Демонстрация: adversarial training с PGD, сравнение устойчивости до/после, упрощённый randomized smoothing.

## 1. Обучение обычной и adversarially-trained моделей

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(0)
np.random.seed(0)

import torchvision
import torchvision.transforms as T

transform = T.Compose([T.ToTensor()])
train_ds = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_ds = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=1, shuffle=True)

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1,16,3,padding=1)
        self.conv2 = nn.Conv2d(16,32,3,padding=1)
        self.fc = nn.Linear(32*7*7,10)
    def forward(self,x):
        x = F.relu(self.conv1(x)); x = F.max_pool2d(x,2)
        x = F.relu(self.conv2(x)); x = F.max_pool2d(x,2)
        x = x.view(x.size(0),-1)
        return self.fc(x)

model = SimpleCNN()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for i,(xb,yb) in enumerate(train_loader):
    opt.zero_grad(); loss = F.cross_entropy(model(xb), yb); loss.backward(); opt.step()
    if i>=300: break
print("Базовая модель обучена, loss:", loss.item())

def pgd_attack(model, x, y, eps=0.2, alpha=0.02, iters=10):
    x_orig = x.clone().detach()
    x_adv = x_orig + torch.empty_like(x_orig).uniform_(-eps, eps)
    x_adv = torch.clamp(x_adv, 0, 1).detach()
    for _ in range(iters):
        x_adv.requires_grad_(True)
        loss = F.cross_entropy(model(x_adv), y)
        grad = torch.autograd.grad(loss, x_adv)[0]
        x_adv = x_adv.detach() + alpha*grad.sign()
        x_adv = torch.max(torch.min(x_adv, x_orig+eps), x_orig-eps)
        x_adv = torch.clamp(x_adv, 0, 1)
    return x_adv.detach()

robust_model = SimpleCNN()
opt_r = torch.optim.Adam(robust_model.parameters(), lr=1e-3)
for i, (xb, yb) in enumerate(train_loader):
    x_adv = pgd_attack(robust_model, xb, yb, eps=0.15, alpha=0.02, iters=5)
    opt_r.zero_grad()
    loss = F.cross_entropy(robust_model(x_adv), yb)
    loss.backward(); opt_r.step()
    if i>=300: break
print("Adversarially-trained модель обучена, loss:", loss.item())


Базовая модель обучена, loss: 0.11312925815582275
Adversarially-trained модель обучена, loss: 0.43603360652923584


## 2. Сравнение устойчивости стандартной и robust модели против PGD

In [2]:

def evaluate_robust_acc(m, loader, eps, n=200):
    correct = 0; total = 0
    for i, (x,y) in enumerate(loader):
        if i>=n: break
        x_adv = pgd_attack(m, x, y, eps=eps, alpha=0.02, iters=10)
        pred = m(x_adv).argmax(1)
        correct += (pred==y).sum().item(); total += len(y)
    return correct/total

acc_standard = evaluate_robust_acc(model, test_loader, eps=0.15)
acc_robust = evaluate_robust_acc(robust_model, test_loader, eps=0.15)
print(f"Точность стандартной модели под PGD-атакой (eps=0.15): {acc_standard:.2%}")
print(f"Точность adversarially-trained модели под той же атакой: {acc_robust:.2%}")


Точность стандартной модели под PGD-атакой (eps=0.15): 47.00%
Точность adversarially-trained модели под той же атакой: 75.50%


## 3. Упрощённый Randomized Smoothing

In [3]:

def smoothed_predict(model, x, sigma=0.1, n_samples=50):
    votes = torch.zeros(10)
    for _ in range(n_samples):
        noise = torch.randn_like(x)*sigma
        pred = model(torch.clamp(x+noise,0,1)).argmax(1)
        votes[pred] += 1
    return votes.argmax().item(), votes

x, y = next(iter(test_loader))
x_adv = pgd_attack(model, x, y, eps=0.15, alpha=0.02, iters=10)
plain_pred = model(x_adv).argmax(1).item()
smooth_pred, votes = smoothed_predict(model, x_adv)
print(f"Истинный класс: {y.item()}")
print(f"Предсказание без smoothing на атакованном входе: {plain_pred}")
print(f"Предсказание с randomized smoothing: {smooth_pred} (голоса: {votes.numpy()})")


Истинный класс: 9
Предсказание без smoothing на атакованном входе: 7
Предсказание с randomized smoothing: 7 (голоса: [ 0.  0.  0.  0.  0.  0.  0. 50.  0.  0.])
